# 📬 Day 02a: Celery & Redis — Task Queues

---

## 🎯 What You'll Master Today
- Celery architecture and when to use task queues
- Celery setup with Redis as broker
- Task creation, retries, and chaining
- Periodic tasks with Celery Beat
- Monitoring with Flower

---

## 🎭 The Post Office Analogy

Your API is a customer service desk. When someone asks to send a package (heavy task), you don't drive it there yourself — you drop it at the **post office** (message broker). A **mail carrier** (worker) picks it up and delivers it.

```
╔═══════════════════════════════════════════════════════════════════╗
║  CELERY ARCHITECTURE                                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║   FastAPI App ──→ Redis (Broker) ──→ Celery Worker(s)           ║
║       │               │                   │                      ║
║   Sends task      Message Queue       Executes task              ║
║   Returns         Holds tasks         Stores result              ║
║   task_id         until picked up     in Redis/DB                ║
║                                                                   ║
║   WHEN TO USE:                                                    ║
║   ✅ Email sending, PDF generation, image processing            ║
║   ✅ Long-running tasks (>1 second)                              ║
║   ✅ Scheduled/periodic tasks (cron-like)                        ║
║   ✅ Tasks that need retry logic                                 ║
║   ❌ Simple 100ms operations → use BackgroundTasks              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Celery Setup (Reference — runs outside notebook)
# ============================================================

print("""
# Install:
pip install celery[redis] redis

# Project structure:
project/
├── app/
│   ├── main.py          # FastAPI app
│   └── api/
├── worker/
│   ├── celery_app.py    # Celery instance
│   └── tasks.py         # Task definitions
├── docker-compose.yml   # Redis + worker
└── requirements.txt
""")

# celery_app.py
print("""
# worker/celery_app.py
from celery import Celery

celery_app = Celery(
    "worker",
    broker="redis://localhost:6379/0",
    backend="redis://localhost:6379/1",
)

celery_app.conf.update(
    task_serializer="json",
    accept_content=["json"],
    result_serializer="json",
    timezone="UTC",
    task_track_started=True,
    task_acks_late=True,           # Ack after task completes (reliable)
    worker_prefetch_multiplier=1,  # One task at a time per worker
)
""")

In [ ]:
# ============================================================
# Defining Tasks
# ============================================================

print("""
# worker/tasks.py
from worker.celery_app import celery_app
import time

# Basic task
@celery_app.task
def send_email(to: str, subject: str, body: str):
    \"\"\"Send an email (simulated).\"\"\"  
    time.sleep(3)  # Simulate email sending
    print(f"Email sent to {to}: {subject}")
    return {"status": "sent", "to": to}

# Task with retry
@celery_app.task(
    bind=True,                  # Gets 'self' (task instance)
    max_retries=3,
    default_retry_delay=60,     # Wait 60s before retry
)
def process_payment(self, order_id: int, amount: float):
    try:
        # Call payment gateway
        result = call_payment_api(order_id, amount)
        return {"status": "success", "transaction_id": result.id}
    except PaymentGatewayError as exc:
        # Retry with exponential backoff
        raise self.retry(
            exc=exc,
            countdown=2 ** self.request.retries * 60  # 60s, 120s, 240s
        )

# Task with time limit
@celery_app.task(time_limit=300, soft_time_limit=240)
def generate_report(report_type: str):
    \"\"\"Generate a large report. Hard limit: 5min, soft limit: 4min.\"\"\"  
    # At soft_time_limit → SoftTimeLimitExceeded raised
    # At time_limit → task killed
    ...
""")

In [ ]:
# ============================================================
# FastAPI + Celery Integration
# ============================================================

print("""
# app/main.py
from fastapi import FastAPI
from worker.tasks import send_email, generate_report
from celery.result import AsyncResult

app = FastAPI()

# Trigger a task
@app.post("/send-email")
def trigger_email(to: str, subject: str, body: str):
    task = send_email.delay(to, subject, body)  # .delay() queues it
    return {"task_id": task.id, "status": "queued"}

# Check task status
@app.get("/task/{task_id}")
def get_task_status(task_id: str):
    result = AsyncResult(task_id)
    response = {
        "task_id": task_id,
        "status": result.status,  # PENDING, STARTED, SUCCESS, FAILURE
    }
    if result.ready():
        response["result"] = result.get()  # Get return value
    return response

# Calling with apply_async for more control
@app.post("/generate-report")
def trigger_report(report_type: str):
    task = generate_report.apply_async(
        args=[report_type],
        countdown=10,      # Delay execution by 10 seconds
        expires=3600,      # Task expires after 1 hour
        queue="reports",   # Send to specific queue
    )
    return {"task_id": task.id}
""")

print("Key methods:")
print("  .delay(args)         → Simple call, queues immediately")
print("  .apply_async(args)   → Full control (countdown, queue, expires)")
print("  AsyncResult(task_id) → Check status, get result")

In [ ]:
# ============================================================
# Task Chaining — Workflow Patterns
# ============================================================

print("""
from celery import chain, group, chord

# CHAIN: task1 → task2 → task3 (sequential, result passed along)
workflow = chain(
    fetch_data.s("url"),         # .s() creates a signature
    process_data.s(),             # Receives fetch_data's result
    save_to_db.s(),               # Receives process_data's result
)
result = workflow.apply_async()

# GROUP: Run tasks in parallel
batch = group(
    send_email.s("a@b.com", "Hi", "Body"),
    send_email.s("c@d.com", "Hi", "Body"),
    send_email.s("e@f.com", "Hi", "Body"),
)
result = batch.apply_async()  # All 3 run in parallel

# CHORD: Group + callback when all done
workflow = chord(
    group(process_item.s(i) for i in range(100)),  # 100 parallel tasks
    aggregate_results.s()  # Called with list of all 100 results
)
""")

print("Workflow patterns:")
print("  chain()  → Sequential pipeline (A → B → C)")
print("  group()  → Parallel execution (A + B + C)")
print("  chord()  → Parallel + callback (A+B+C → D)")

In [ ]:
# ============================================================
# Periodic Tasks — Celery Beat
# ============================================================

print("""
# worker/celery_app.py — add this
from celery.schedules import crontab

celery_app.conf.beat_schedule = {
    "cleanup-expired-tokens": {
        "task": "worker.tasks.cleanup_tokens",
        "schedule": crontab(minute=0, hour="*/6"),  # Every 6 hours
    },
    "daily-report": {
        "task": "worker.tasks.generate_daily_report",
        "schedule": crontab(minute=0, hour=9),       # 9 AM daily
    },
    "health-check": {
        "task": "worker.tasks.health_check",
        "schedule": 30.0,  # Every 30 seconds
    },
}

# Run beat scheduler:
# celery -A worker.celery_app beat --loglevel=info

# Run worker:
# celery -A worker.celery_app worker --loglevel=info --concurrency=4

# Monitor with Flower:
# pip install flower
# celery -A worker.celery_app flower --port=5555
""")

print("Crontab examples:")
print("  crontab(minute=0, hour=0)          → Midnight daily")
print("  crontab(minute=0, hour='*/2')       → Every 2 hours")
print("  crontab(minute=0, hour=0, day_of_week='mon') → Monday midnight")
print("  30.0                                → Every 30 seconds")

In [ ]:
# ============================================================
# Docker Compose for Celery Stack
# ============================================================

print("""
# docker-compose.yml
services:
  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"

  api:
    build: .
    command: uvicorn app.main:app --host 0.0.0.0 --port 8000
    ports:
      - "8000:8000"
    depends_on:
      - redis
    environment:
      - CELERY_BROKER_URL=redis://redis:6379/0

  worker:
    build: .
    command: celery -A worker.celery_app worker --loglevel=info --concurrency=4
    depends_on:
      - redis
    environment:
      - CELERY_BROKER_URL=redis://redis:6379/0

  beat:
    build: .
    command: celery -A worker.celery_app beat --loglevel=info
    depends_on:
      - redis

  flower:
    build: .
    command: celery -A worker.celery_app flower --port=5555
    ports:
      - "5555:5555"
    depends_on:
      - redis
""")

---

## 🗺️ Celery Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  CELERY DECISION GUIDE                                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  WHEN TO USE:                                                    ║
║    Task > 1 second          → Celery                            ║
║    Needs retry/scheduling   → Celery                            ║
║    Quick fire-and-forget    → BackgroundTasks                   ║
║    Simple async I/O         → async/await                       ║
║                                                                   ║
║  TASK CALLING:                                                   ║
║    .delay(args)      → simple, immediate                        ║
║    .apply_async()    → countdown, queue, expires                ║
║    .s()              → signature (for workflows)                ║
║                                                                   ║
║  WORKFLOWS:                                                      ║
║    chain()  → A → B → C                                        ║
║    group()  → A + B + C  (parallel)                             ║
║    chord()  → (A+B+C) → D  (parallel + callback)               ║
║                                                                   ║
║  RELIABILITY:                                                    ║
║    task_acks_late=True  → ack after completion                  ║
║    max_retries + retry()→ automatic retry with backoff         ║
║    time_limit           → kill stuck tasks                      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is Celery? | Distributed task queue for Python. Uses broker (Redis/RabbitMQ) + workers |
| 2 | When to use Celery vs BackgroundTasks? | Celery: heavy/long tasks, retries, scheduling. BG: lightweight, quick, in-process |
| 3 | How do retries work? | bind=True, self.retry(exc=exc, countdown=N), max_retries, exponential backoff |
| 4 | chain vs group vs chord? | chain: sequential. group: parallel. chord: parallel + final callback |
| 5 | What is Celery Beat? | Scheduler for periodic tasks (cron-like). Runs as separate process |
| 6 | How to monitor Celery? | Flower web UI, celery inspect active/reserved/revoked |
| 7 | Redis vs RabbitMQ as broker? | Redis: simpler, good enough. RabbitMQ: more features (priority, routing) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Celery = distributed task queue (broker + workers)   │
## │  • .delay() for simple calls, .apply_async() for control│
## │  • Use bind=True + self.retry() for resilient tasks     │
## │  • chain/group/chord for complex workflows              │
## │  • Celery Beat for periodic/scheduled tasks             │
## │  • task_acks_late=True for reliability                  │
## │  • Flower for monitoring                                │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **WebSockets** — Real-time bidirectional communication